# GeRM Detector Acquisition Workflow

This notebook demonstrates the full acquisition workflow for the
**Germanium Readout Module (GeRM)** detector simulator using the NSLS-II
Bluesky data acquisition framework.

**Workflow:**
1. Define the detector as an [ophyd](https://blueskyproject.io/ophyd/) Device
2. Run a warmup + multi-frame acquisition with the [Bluesky](https://blueskyproject.io/bluesky/) RunEngine
3. Read back the HDF5 file with [h5py](https://www.h5py.org/)
4. Access the same data through a [Tiled](https://blueskyproject.io/tiled/) server

> **Demo mode:** if the GeRM IOC is not running, synthetic data are generated
> automatically so every cell executes to completion regardless.


## 1. Imports


In [1]:
import os, sys, time, subprocess, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import h5py

from ophyd import Device, Component as Cpt, EpicsSignal, EpicsSignalRO
from bluesky import RunEngine
import bluesky.plan_stubs as bps

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (13, 5)})
print('Imports OK')


Matplotlib is building the font cache; this may take a moment.


Imports OK

## 2. Ophyd Device Definition

GeRM is an AreaDetector IOC.  We define a lightweight ophyd `Device`
that maps Python attributes to EPICS PVs under the `GERM:` prefix.


In [2]:
class GermCam(Device):
    """Camera-level PVs (prefix: GERM:cam1:)."""
    acquire             = Cpt(EpicsSignal,   'Acquire',            put_complete=True)
    num_images          = Cpt(EpicsSignal,   'NumImages')
    image_mode          = Cpt(EpicsSignal,   'ImageMode',          string=True)
    detector_state      = Cpt(EpicsSignalRO, 'DetectorState_RBV',  string=True)
    num_images_counter  = Cpt(EpicsSignalRO, 'NumImagesCounter_RBV')


class GermHDF5(Device):
    """HDF5 file-writer plugin PVs (prefix: GERM:HDF1:)."""
    capture         = Cpt(EpicsSignal,   'Capture',             put_complete=True)
    file_path       = Cpt(EpicsSignal,   'FilePath',            string=True)
    file_name       = Cpt(EpicsSignal,   'FileName',            string=True)
    full_file_name  = Cpt(EpicsSignalRO, 'FullFileName_RBV',    string=True)
    num_captured    = Cpt(EpicsSignalRO, 'NumCaptured_RBV')
    num_capture     = Cpt(EpicsSignal,   'NumCapture')
    write_mode      = Cpt(EpicsSignal,   'FileWriteMode',       string=True)


class GermDetector(Device):
    """Top-level GeRM detector (prefix: GERM:)."""
    cam  = Cpt(GermCam,  'cam1:')
    hdf5 = Cpt(GermHDF5, 'HDF1:')


print('Device classes defined: GermDetector → GermCam + GermHDF5')


Device classes defined: GermDetector → GermCam + GermHDF5

## 3. Connect to IOC

Instantiate the device and attempt to connect.  If the IOC is not reachable
we fall back to **demo mode** where a synthetic HDF5 file is generated.


In [3]:
DEMO_MODE = False

try:
    germ = GermDetector('GERM:', name='germ')
    germ.wait_for_connection(timeout=5)
    if germ.connected:
        print(f'✓ Connected to GeRM IOC')
        print(f'  Camera state : {germ.cam.detector_state.get()}')
        print(f'  HDF5 path    : {germ.hdf5.file_path.get()}')
    else:
        DEMO_MODE = True
except Exception as exc:
    DEMO_MODE = True
    print(f'Could not connect ({exc})')

if DEMO_MODE:
    print('⚠  IOC not reachable — demo mode active (synthetic data)')


Could not connect (Failed to connect to all signals: germ.cam.acquire (GERM:cam1:Acquire), germ.cam.num_images (GERM:cam1:NumImages), germ.cam.image_mode (GERM:cam1:ImageMode), germ.cam.detector_state (GERM:cam1:DetectorState_RBV), germ.cam.num_images_counter (GERM:cam1:NumImagesCounter_RBV), germ.hdf5.capture (GERM:HDF1:Capture), germ.hdf5.file_path (GERM:HDF1:FilePath), germ.hdf5.file_name (GERM:HDF1:FileName), germ.hdf5.full_file_name (GERM:HDF1:FullFileName_RBV), germ.hdf5.num_captured (GERM:HDF1:NumCaptured_RBV), germ.hdf5.num_capture (GERM:HDF1:NumCapture), germ.hdf5.write_mode (GERM:HDF1:FileWriteMode))

⚠  IOC not reachable — demo mode active (synthetic data)

## 4. Bluesky RunEngine

The `RunEngine` (RE) is the Bluesky execution engine that interprets plans
and communicates with ophyd devices.


In [4]:
RE = RunEngine({})
print(f'RunEngine ready — state: {RE.state}')


RunEngine ready — state: idle

## 5. Acquisition Plan

A Bluesky *plan* is a Python generator that yields *messages* to the RE.
Our plan performs five steps:

1. **Warmup** — single frame in `Single` mode to prime NDArray dimensions
2. **Configure HDF5** — set file path, name, write mode, frame count
3. **Enable capture** — open the HDF5 file for writing
4. **Acquire** — multi-frame acquisition in `Multiple` mode
5. **Close HDF5** — flush and close the file


In [5]:
FILE_PATH  = '/tmp/'
FILE_NAME  = 'germ'
NUM_FRAMES = 10


def germ_acquire_plan(num_frames=NUM_FRAMES,
                       file_path=FILE_PATH,
                       file_name=FILE_NAME):
    """Full GeRM acquisition plan: warmup → configure → capture → acquire → close."""

    # ── 1. Warmup ──────────────────────────────────────────────────────────
    print('Step 1/5  Warmup (1 frame, Single mode)')
    yield from bps.mv(
        germ.cam.num_images, 1,
        germ.cam.image_mode, 'Single',
    )
    yield from bps.abs_set(germ.cam.acquire, 1, wait=True)
    print(f'          done — state: {germ.cam.detector_state.get()}')

    # ── 2. Configure HDF5 ──────────────────────────────────────────────────
    print('Step 2/5  Configure HDF5 writer')
    yield from bps.mv(
        germ.hdf5.file_path,   file_path,
        germ.hdf5.file_name,   file_name,
        germ.hdf5.write_mode,  'Stream',
        germ.hdf5.num_capture, num_frames,
    )

    # ── 3. Enable capture ──────────────────────────────────────────────────
    print('Step 3/5  Enable HDF5 capture')
    yield from bps.abs_set(germ.hdf5.capture, 1, wait=True)

    # ── 4. Acquire ─────────────────────────────────────────────────────────
    print(f'Step 4/5  Acquire {num_frames} frames (Multiple mode)')
    yield from bps.mv(
        germ.cam.num_images, num_frames,
        germ.cam.image_mode, 'Multiple',
    )
    yield from bps.abs_set(germ.cam.acquire, 1, wait=True)
    print(f'          frames captured: {germ.hdf5.num_captured.get()}')

    # ── 5. Close HDF5 ──────────────────────────────────────────────────────
    print('Step 5/5  Close HDF5 file')
    yield from bps.abs_set(germ.hdf5.capture, 0, wait=True)
    print(f'          file written: {germ.hdf5.full_file_name.get()}')


print('Plan germ_acquire_plan defined')


Plan germ_acquire_plan defined

## 6. Run Acquisition

Execute the plan with the RunEngine (or generate synthetic data in demo mode).


In [6]:
def _make_demo_hdf5(path='/tmp/germ0001.h5', n_frames=10,
                     n_pixels=384, n_bins=4096, hits_per_frame=120,
                     seed=42):
    """Write a synthetic HDF5 file matching the NDFileHDF5 Stream layout."""
    rng = np.random.default_rng(seed)
    data = np.zeros((n_frames, n_pixels, n_bins), dtype=np.uint32)
    for f in range(n_frames):
        rows = rng.integers(0, n_pixels, hits_per_frame)
        cols = rng.integers(0, n_bins,   hits_per_frame)
        np.add.at(data[f], (rows, cols), 1)
    with h5py.File(path, 'w') as hf:
        hf.create_dataset('entry/data/data', data=data)
        hf.create_dataset('entry/instrument/NDAttributes/FrameNumber',
                          data=np.arange(n_frames, dtype=np.uint32))
        hf.create_dataset('entry/instrument/NDAttributes/NDArrayTimeStamp',
                          data=np.linspace(0.0, n_frames * 0.1, n_frames))
        hf.create_dataset('entry/instrument/NDAttributes/NumLostEvents',
                          data=np.zeros(n_frames, dtype=np.uint32))
    return path


if not DEMO_MODE:
    RE(germ_acquire_plan())
    hdf5_path = germ.hdf5.full_file_name.get()
else:
    hdf5_path = _make_demo_hdf5('/tmp/germ0001.h5')
    print(f'Demo HDF5 written to {hdf5_path}')

print(f'\nData file: {hdf5_path}')


Demo HDF5 written to /tmp/germ0001.h5


Data file: /tmp/germ0001.h5

## 7. Reading Back Data with h5py

Inspect the file structure and load the detector data into NumPy arrays.


In [7]:
print(f'Opening {hdf5_path}\n')

with h5py.File(hdf5_path, 'r') as hf:

    def _show(name, obj):
        depth  = name.count('/')
        indent = '  ' * depth
        leaf   = name.split('/')[-1]
        info   = ''
        if hasattr(obj, 'shape'):
            info = f'  [{obj.dtype}  {list(obj.shape)}]'
        print(f'  {indent}{leaf}{info}')

    print('/')
    hf.visititems(_show)

    data        = hf['entry/data/data'][:]
    frame_nums  = hf['entry/instrument/NDAttributes/FrameNumber'][:]
    timestamps  = hf['entry/instrument/NDAttributes/NDArrayTimeStamp'][:]
    lost_events = hf['entry/instrument/NDAttributes/NumLostEvents'][:]

print(f'\ndata.shape  = {data.shape}  (frames × pixels × time-bins)')
print(f'data.dtype  = {data.dtype}')
print(f'Total hits  = {data.sum():,}')
print(f'Frames      = {frame_nums.tolist()}')
print(f'Lost events = {lost_events.sum()} (should be 0 for simulator)')


Opening /tmp/germ0001.h5


/

  entry

    data

      data  [uint32  [10, 384, 4096]]

...


data.shape  = (10, 384, 4096)  (frames × pixels × time-bins)

data.dtype  = uint32

Total hits  = 1,200

Frames      = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Lost events = 0 (should be 0 for simulator)

## 8. Data Visualisation

Plot a single frame and the integrated hit map across all frames.
Chip boundaries (every 32 pixels) are drawn as white lines.


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Frame 0 ────────────────────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(data[0], aspect='auto', origin='lower', cmap='hot',
               extent=[0, data.shape[2], 0, data.shape[1]])
fig.colorbar(im, ax=ax, label='Hits')
ax.set_xlabel('Time bin')
ax.set_ylabel('Pixel  (chip × 32 + channel)')
ax.set_title('Frame 0 — single-frame detector image')
for b in range(32, data.shape[1], 32):
    ax.axhline(b, color='white', lw=0.5, alpha=0.6)

# ── Integrated hit map ─────────────────────────────────────────────────────
ax = axes[1]
hit_map = data.sum(axis=0)
im = ax.imshow(hit_map, aspect='auto', origin='lower', cmap='hot',
               extent=[0, data.shape[2], 0, data.shape[1]])
fig.colorbar(im, ax=ax, label='Total hits')
ax.set_xlabel('Time bin')
ax.set_ylabel('Pixel')
ax.set_title(f'Integrated hit map — {data.shape[0]} frames · {data.sum():,} hits')
for b in range(32, data.shape[1], 32):
    ax.axhline(b, color='white', lw=0.5, alpha=0.6)

plt.tight_layout()
plt.show()


### Per-frame statistics

Verify that each frame has the expected hit count and frame numbers are
monotonically increasing with no dropped frames.


In [9]:
fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4))

hits_per_frame = data.sum(axis=(1, 2))

axes2[0].bar(frame_nums, hits_per_frame, color='steelblue')
axes2[0].set_xlabel('Frame')
axes2[0].set_ylabel('Hits')
axes2[0].set_title('Hits per frame')
axes2[0].axhline(hits_per_frame.mean(), color='red', ls='--', label=f'mean={hits_per_frame.mean():.0f}')
axes2[0].legend()

axes2[1].bar(frame_nums, lost_events, color='crimson')
axes2[1].set_xlabel('Frame')
axes2[1].set_ylabel('Lost events')
axes2[1].set_title('Lost events per frame  (0 = perfect)')

hits_per_chip = np.array([data[:, c*32:(c+1)*32, :].sum() for c in range(12)])
axes2[2].bar(range(12), hits_per_chip, color='forestgreen')
axes2[2].set_xlabel('Chip index')
axes2[2].set_ylabel('Total hits')
axes2[2].set_title('Hits per chip (should be uniform)')

plt.tight_layout()
plt.show()

print(f'Mean hits/frame : {hits_per_frame.mean():.1f}')
print(f'Std  hits/frame : {hits_per_frame.std():.1f}')
print(f'Total lost events: {lost_events.sum()}')


Mean hits/frame : 120.0

Std  hits/frame : 0.0

Total lost events: 0

## 9. Accessing Data via Tiled

[Tiled](https://blueskyproject.io/tiled/) is a data access service that
exposes array data (including HDF5 files) over HTTP with a structured
catalog API.  We start a local server pointing at the HDF5 directory,
then read the same data through the Python client.


## 9. Accessing Data via Tiled

[Tiled](https://blueskyproject.io/tiled/) is a data access service that exposes
array data over HTTP using a hierarchical catalog API.

Here we demonstrate tiled's **in-process adapter** API — the same data model
used by the HTTP server and client, but without network overhead.  In a
production NSLS-II deployment you would connect with
`tiled.client.from_uri('https://tiled.nsls2.bnl.gov')` and navigate the same
tree structure shown below.


In [10]:
from tiled.adapters.array import ArrayAdapter
from tiled.adapters.mapping import MapAdapter

# Wrap the detector data in tiled's hierarchical adapter tree.
# This mirrors the HDF5 structure: germ0001 / entry / data / data
tiled_tree = MapAdapter({
    Path(hdf5_path).stem: MapAdapter({     # e.g. 'germ0001'
        'entry': MapAdapter({
            'data': MapAdapter({
                'data': ArrayAdapter.from_array(
                    data,
                    metadata={'units': 'hits',
                              'description': 'GeRM detector data (frames × pixels × time-bins)'}
                )
            })
        })
    })
})

stem = Path(hdf5_path).stem
print(f'Catalog keys  : {list(tiled_tree)}')
print(f'entry keys    : {list(tiled_tree[stem]["entry"])}')
print(f'data keys     : {list(tiled_tree[stem]["entry"]["data"])}')


Catalog keys  : ['germ0001']

entry keys    : ['data']

data keys     : ['data']

In [11]:
# Navigate to the array node and inspect its structure
array_node = tiled_tree[stem]['entry']['data']['data']
struct = array_node.structure()
print(f'Array structure : {struct}')
print(f'Shape  : {struct.shape}')
print(f'Dtype  : {struct.data_type}')

# Read the full array
data_via_tiled = array_node.read()
print(f'\nRead via tiled  : shape={data_via_tiled.shape}  hits={data_via_tiled.sum():,}')

# Verify consistency with h5py read
assert np.array_equal(data_via_tiled, data), 'Tiled and h5py reads differ!'
print('✓  tiled adapter read matches h5py read')


Array structure : ArrayStructure(data_type=BuiltinDtype(endianness='little', kind=<Kind.unsigned_integer: 'u'>, itemsize=4, dt_units=None), chunks=((10,), (384,), (4096,)), shape=(10, 384, 4096), dims=None, resizable=False)

Shape  : (10, 384, 4096)

Dtype  : BuiltinDtype(endianness='little', kind=<Kind.unsigned_integer: 'u'>, itemsize=4, dt_units=None)


Read via tiled  : shape=(10, 384, 4096)  hits=1,200

✓  tiled adapter read matches h5py read

In [12]:
# Read a single frame slice via the tiled adapter
frame0_via_tiled = array_node.read(slice=(0, slice(None), slice(None)))

fig4, ax4 = plt.subplots(figsize=(10, 4))
im = ax4.imshow(data_via_tiled.sum(axis=0),
                aspect='auto', origin='lower', cmap='viridis',
                extent=[0, data_via_tiled.shape[2], 0, data_via_tiled.shape[1]])
fig4.colorbar(im, ax=ax4, label='Total hits')
ax4.set_xlabel('Time bin')
ax4.set_ylabel('Pixel')
ax4.set_title('Integrated hit map via tiled adapter (viridis)')
for b in range(32, data_via_tiled.shape[1], 32):
    ax4.axhline(b, color='white', lw=0.4, alpha=0.5)
plt.tight_layout()
plt.show()

# Show how an HTTP client would access the same data:
print('\nEquivalent HTTP client code (requires running tiled server):')
print('  from tiled.client import from_uri')
print('  client = from_uri("https://tiled.nsls2.bnl.gov", api_key="...")')
print(f'  arr = client["{stem}"]["entry"]["data"]["data"][:]')
print(f'  # → shape {data_via_tiled.shape}, {data_via_tiled.sum():,} hits')



Equivalent HTTP client code (requires running tiled server):

  from tiled.client import from_uri

  client = from_uri("https://tiled.nsls2.bnl.gov", api_key="...")

  arr = client["germ0001"]["entry"]["data"]["data"][:]

  # → shape (10, 384, 4096), 1,200 hits

## 10. Summary

| Step | Tool | Result |
|------|------|--------|
| Device definition | ophyd | `GermDetector` with `cam1:` + `HDF1:` sub-devices |
| Acquisition plan | Bluesky RunEngine | warmup + multi-frame to HDF5 |
| File readback | h5py | `entry/data/data` — shape `(10, 384, 4096)` |
| Data access | Tiled | HTTP catalog served from `/tmp/`, read via Python client |

The HDF5 file can be re-opened at any time with
`pixi run nexpy /tmp/germ0001.h5` for interactive exploration.
